# **Data Scrapping and Prepocessing**

In [2]:
# Import Library

from google_play_scraper import reviews, Sort
import pandas as pd

In [4]:
from google_play_scraper import reviews, Sort
import pandas as pd
import time

# ID aplikasi Duolingo
app_id = 'com.duolingo'

# Daftar kode bahasa yang ingin di-scrape
languages = ['en', 'id']

# Menyimpan semua ulasan
all_reviews = []

for lang in languages:
    print(f"Scraping reviews for language: {lang}")
    try:
        result, _ = reviews(
            app_id,
            lang=lang,
            country='us',  # Kamu dapat menyesuaikan negara sesuai kebutuhan
            sort=Sort.NEWEST,
            count=1000,
            filter_score_with=None
        )
        for review in result:
            review['language'] = lang
        all_reviews.extend(result)
        time.sleep(1)  # Menghindari pembatasan dari server
    except Exception as e:
        print(f"Error scraping language {lang}: {e}")

Scraping reviews for language: en
Scraping reviews for language: id


In [5]:
# Simpan ke DataFrame dan ekspor ke CSV
df = pd.DataFrame(all_reviews)
df.to_csv('data_1.csv', index=False, encoding='utf-8-sig')

print("Selesai! Semua ulasan telah disimpan.")

Selesai! Semua ulasan telah disimpan.


In [2]:
import pandas as pd
df = pd.read_csv(r"D:\Internship\AIBeecara\PROJECT TEST\data_1.csv")
df.head(3)

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion,language
0,d3b962c4-6145-4f8a-aa68-57cecfe86b13,Alex Nelson,https://play-lh.googleusercontent.com/a-/ALV-U...,I have been using this app(free version) for a...,1,471,6.26.2,2025-04-24 16:46:19,NaN,NaN,6.26.2,en
1,67466d15-b34c-45d8-aa34-e3c896b52877,Jared Lindell,https://play-lh.googleusercontent.com/a/ACg8oc...,Payed version ≈ $100 per year. if you want to ...,2,397,6.25.4,2025-04-11 03:24:37,NaN,NaN,6.25.4,en
2,91a3e121-d00f-47b3-a40b-99299d24179e,Master Kayla Rommann,https://play-lh.googleusercontent.com/a-/ALV-U...,Editing after the latest update. The Heart sit...,3,128,6.27.4,2025-04-24 19:17:11,NaN,NaN,6.27.4,en


In [3]:
data_coment = df.drop(columns=['reviewId', 'userName','userImage', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion', 'language' ])

In [4]:
data_coment.head()

,content
0,I have been using this app(free version) for a...
1,Payed version ≈ $100 per year. if you want to ...
2,Editing after the latest update. The Heart sit...
3,This app presents basic conversation via repet...
4,Great app. Wish they had their ad situation fi...


### **CASEFOLDING**
(lowercase)

In [5]:
data_coment["casefolding"]= data_coment['content'].str.lower()
data_coment.head(3)

,content,casefolding
0,I have been using this app(free version) for a...,i have been using this app(free version) for a...
1,Payed version ≈ $100 per year. if you want to ...,payed version ≈ $100 per year. if you want to ...
2,Editing after the latest update. The Heart sit...,editing after the latest update. the heart sit...


### **Text Cleaning/Normalization**
(regular expression)

In [7]:
import re
import string

def preprocess_text(text):
    # Menghapus teks dalam tanda kurung siku [re ...]
    text = re.sub(r'\[re\s[^\]]+\]', '', text)
    
    # Menghapus mention
    text = re.sub('@[^\s]+', '', text)
    
    # Menghapus karakter escape
    text = text.replace('\\t', "").replace('\\n', "").replace('\\u', "").replace('\\', "")
    
    # Mengonversi ke ASCII
    text = text.encode('ascii', 'replace').decode('ascii')
    
    # Menghapus hashtag dan URL
    text = ' '.join(re.sub(r'([#@]\S*)|(\w+:\/\/\S+)', " ", text).split())
    
    # Menghapus angka
    text = re.sub(r"\d+", "", text)
    
    # Menghapus tanda baca
    text = text.translate(str.maketrans("", "", string.punctuation))
    
    # Menghapus spasi berlebih
    text = text.strip()
    text = re.sub('\s+', ' ', text)
    
    # Menghapus kata satu huruf
    text = re.sub(r"\b[a-zA-Z]\b", "", text)
    
    # Menghapus kata 'rt' (retweet)
    text = re.sub(r'\brt\b', '', text)
    
    # Menghapus sisa URL
    text = text.replace("http://", " ").replace("https://", " ")
    
    return text

In [8]:
data_coment['text_clean']= data_coment['casefolding'].apply(preprocess_text)
data_coment.head(3)

,content,casefolding,text_clean
0,I have been using this app(free version) for a...,i have been using this app(free version) for a...,have been using this appfree version for few...
1,Payed version ≈ $100 per year. if you want to ...,payed version ≈ $100 per year. if you want to ...,payed version per year if you want to use the ...
2,Editing after the latest update. The Heart sit...,editing after the latest update. the heart sit...,editing after the latest update the heart situ...


### **STOPWORD REMOVAL**
eliminating common words (like "dan", "the," "a," "and") that are frequently used but don't carry much semantic meaning.

In [9]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from langdetect import detect
import string
import nltk

# Download resource
nltk.download('stopwords')
nltk.download('punkt')

# Stopwords bahasa
stop_words_en = set(stopwords.words('english'))
stop_words_id = set(stopwords.words('indonesian'))

# Fungsi untuk stopword removal berdasarkan bahasa
def remove_stopwords_by_language(text):
    try:
        lang = detect(text)
    except:
        lang = 'unknown'
    
    tokens = word_tokenize(text)
    
    if lang == 'id':
        stop_words = stop_words_id
    elif lang == 'en':
        stop_words = stop_words_en
    else:
        stop_words = set()
    
    return ' '.join([w for w in tokens if w.lower() not in stop_words and w not in string.punctuation])

# Terapkan ke kolom tokenize
data_coment['stopword'] = data_coment['text_clean'].apply(remove_stopwords_by_language)
data_coment.head(3)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HAJRAN\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\HAJRAN\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


,content,casefolding,text_clean,stopword
0,I have been using this app(free version) for a...,i have been using this app(free version) for a...,have been using this appfree version for few...,using appfree version years pretty good past m...
1,Payed version ≈ $100 per year. if you want to ...,payed version ≈ $100 per year. if you want to ...,payed version per year if you want to use the ...,payed version per year want use interactive vi...
2,Editing after the latest update. The Heart sit...,editing after the latest update. the heart sit...,editing after the latest update the heart situ...,editing latest update heart situation went ann...


In [11]:
data_coment['stopword'].tail(30)

1970    aplikasi berguna pintar pelajaran bahasa inggr...
1971    cocok banget belajar bahasa asing negaradan po...
1972    belajar bahasa bener bener baguss gak rugi deh...
1973    membantu proses belajar bahasa belajar menarik...
1974    membantu meningkatkan skill bahasa inggris ter...
1975    sumpah aplikasi bagus banget bagus suka banget...
1976    berguna banget coba minggu gratis super duolin...
1977    membantu mempelajari bahasa negara mudah aplik...
1978    apk membantu belajar basic bahasa inggris jump...
1979    seruu kalo pas belajar gak males karakternya l...
1980    duolingo aplikasi belajar bahasa terbaik dunia...
1981    apk terbagus belajar bahasa bicara bahasa ingg...
1982    duolingo membantu belajar pelajaran duolingo n...
1983    karna duo lingo mengajarkan bahasa inggrisaku ...
1984    aplikasi bagus orangorang yg belajar bahasa ap...
1985    bagus duolingo ak jdi mengerti lumayan padam a...
1986    disni seru banget baja komentar jang ragu undu...
1987    duolin

### **STEMMING**
reduces words to their base or root form, also known as the stem

In [12]:
from nltk.stem import PorterStemmer
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from langdetect import detect
from nltk.tokenize import word_tokenize
import nltk

# Download resource NLTK
nltk.download('punkt')

# Inisialisasi stemmer
stemmer_en = PorterStemmer()

factory = StemmerFactory()
stemmer_id = factory.create_stemmer()

# Fungsi stemming berdasarkan bahasa
def stem_by_language(text):
    try:
        lang = detect(text)
    except:
        lang = 'unknown'
    
    tokens = word_tokenize(text)

    if lang == 'id':
        # Sastrawi expects a full sentence, not per token
        return stemmer_id.stem(text)
    elif lang == 'en':
        return ' '.join([stemmer_en.stem(w) for w in tokens])
    else:
        return text  # Jika bahasa tidak dikenali, dikembalikan apa adanya

# Terapkan ke kolom hasil stopword removal sebelumnya (misalnya 'cleaned')
data_coment['stemmed'] = data_coment['stopword'].apply(stem_by_language)

# Cetak hasil
data_coment.head(3)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\HAJRAN\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


,content,casefolding,text_clean,stopword,stemmed
0,I have been using this app(free version) for a...,i have been using this app(free version) for a...,have been using this appfree version for few...,using appfree version years pretty good past m...,use appfre version year pretti good past month...
1,Payed version ≈ $100 per year. if you want to ...,payed version ≈ $100 per year. if you want to ...,payed version per year if you want to use the ...,payed version per year want use interactive vi...,pay version per year want use interact video c...
2,Editing after the latest update. The Heart sit...,editing after the latest update. the heart sit...,editing after the latest update the heart situ...,editing latest update heart situation went ann...,edit latest updat heart situat went annoy stup...


In [13]:
# Simpan ke DataFrame dan ekspor ke CSV
df = data_coment
df.to_csv('data_stemm.csv', index=False)

print("Selesai! Semua ulasan telah disimpan.")

Selesai! Semua ulasan telah disimpan.


### **Tokenizing**
(process of breaking down a text into smaller units called tokens, which can be words, sentences, or even characters)

In [ ]:
from nltk.tokenize import word_tokenize

def word_tokenize_wrapper(cleaning):
  return word_tokenize(cleaning)

In [ ]:
data_coment['tokenize']= data_coment['text_clean'].apply(word_tokenize_wrapper)
data_coment.head(3)